# Importaciones

In [1]:
include("dependencies.jl")
include("helpers.jl")
include("wrappers.jl")

# Preparación de los datos (20%)

## 1. Carga y unificación de los datos

In [2]:
csv_inv_a = glob("Investigador A/day */*.csv", DATA_PATH)
csv_inv_b = glob("Investigador B/*.csv", DATA_PATH)
all_csv = vcat(csv_inv_a, csv_inv_b)
dfs = [CSV.read(file, DataFrame) for file in all_csv]
df_total = vcat(dfs...);

## 2. Análisis de valores ausentes

### Para analizar la calidad del dataset, se evaluó la presencia de valores ausentes tanto a nivel global como por variable. El porcentaje total de celdas con valores nulos en el dataset (`df_total`) es cercano al **1%** A nivel de columnas, **175 de las 563 variables** presentan al menos un valor faltante. Las variables con mayor proporción de valores ausentes alcanzan ligeramente más del **10%**.

In [3]:
function nulls_analysis(df::DataFrame; top=10)

    # Porcentaje total de nulos
    total_missing = sum(count(ismissing, df[!, col]) for col in names(df))
    total_values  = nrow(df) * ncol(df)
    pct_total = (total_missing / total_values) * 100
    println("Porcentaje de valores ausentes en el dataset: $(pct_total) %")

    # Porcentaje de nulos por columna
    df_nulls = DataFrame(
        Variable = names(df),
        NullsPercentage = [mean(ismissing.(df[!, c])) * 100 for c in names(df)]
    )

    # Variables con algún valor nulo
    df_nulls_pos = filter(:NullsPercentage => p -> p > 0, df_nulls)
    println("Variables con algún valor ausente: ", nrow(df_nulls_pos))

    # Ordenar de mayor a menor
    sort!(df_nulls_pos, :NullsPercentage, rev=true)

    # Mostrar top variables con más nulos
    if nrow(df_nulls_pos) > 0
        n_show = min(top, nrow(df_nulls_pos))
        println("\nTop $n_show variables con valores ausentes:")
        pretty_table(first(df_nulls_pos, n_show))
    else
        println("Ninguna columna contiene valores ausentes.")
    end

    return df_nulls_pos, pct_total
end

nulls_analysis(df_total);

Porcentaje de valores ausentes en el dataset: 0.9984242033534787 %
Variables con algún valor ausente: 175

Top 10 variables con valores ausentes:
┌──────────────────────────┬─────────────────┐
│                 Variable │ NullsPercentage │
│                   String │         Float64 │
├──────────────────────────┼─────────────────┤
│       tBodyGyroMag-mad() │         10.0301 │
│       tBodyGyroMag-iqr() │         10.0301 │
│         fBodyAcc-mad()-Y │         10.0204 │
│    fBodyAccJerk-mean()-X │         10.0204 │
│  tBodyAccJerk-energy()-X │          10.001 │
│ tBodyAccJerk-entropy()-Y │          10.001 │
│        tBodyAccMag-max() │          10.001 │
│     tGravityAccMag-std() │          10.001 │
│ tGravityAccMag-entropy() │          10.001 │
│         fBodyAcc-std()-X │          10.001 │
└──────────────────────────┴─────────────────┘


## 3. Tratamiento y transformación de datos

### En esta sección preparamos el conjunto de datos para su uso en los algoritmos de clasificación. El objetivo es obtener un dataset completamente limpio, sin valores ausentes y con la variable objetivo correctamente codificada. Queremos mantener una versión sin modificar del dataset, por lo que creamos una copia independiente llamada **df_imputed**. Como los datos pertenecen a diferentes individuos, la imputación se realiza por sujeto.

In [4]:
df_imputed = deepcopy(df_total)

function impute_feature(df::DataFrame)
    individuals = groupby(df, :subject)

    for individual in individuals
        for col in names(individual)

            if col in (:subject, :Activity)
                continue
            end
        
            col_data = individual[!, col]

            if eltype(skipmissing(col_data)) <: Number #  Solo imputamos variables numéricas
                med = median(skipmissing(col_data))
                replace!(col_data, missing => med)
            end
        end
    end

    return df
end

impute_feature(df_imputed)
nulls_analysis(df_imputed); # Verificamos que ya no hay nulos

# La etiqueta debe ser categórica
df_imputed.Activity = categorical(df_imputed.Activity)

# Separamos features y target (características y etiqueta)
y = df_imputed.Activity
x = DataFrames.select(df_imputed, Not([:subject, :Activity]))

# Mostramos que estos tratamientos se aplican correctamente
println("\nTipo de la variable objetivo (y): $(eltype(df_imputed.Activity))")
println("Número de features (columnas en dataset de features): ", ncol(x))

Porcentaje de valores ausentes en el dataset: 0.0 %
Variables con algún valor ausente: 0
Ninguna columna contiene valores ausentes.

Tipo de la variable objetivo (y): CategoricalArrays.CategoricalValue{String31, UInt32}
Número de features (columnas en dataset de features): 561


## 4. Partición Holdout

### Para garantizar una evaluación estrictamente independiente, se realiza una partición hold-out basada en sujetos completos. El 10% de los individuos se reserva como test, y el 90% restante se usa para entrenamiento + validación. Esta división evita que datos del mismo sujeto aparezcan en distintos conjuntos, eliminando cualquier riesgo de data leakage.

In [ ]:
# Crear particiones
subjects = unique(df_imputed.subject)
Random.seed!(SEED)
shuffle!(subjects)
n_test = round(Int, length(subjects) * 0.10)
test_subjects = subjects[1:n_test]
trainval_subjects = subjects[n_test+1:end]

df_trainval = filter(row -> row.subject in trainval_subjects, df_imputed)
df_test = filter(row -> row.subject in test_subjects, df_imputed)

# Preparar variables finales
y_trainval = df_trainval.Activity
X_trainval = DataFrames.select(df_trainval, Not([:subject, :Activity]))
folds_trainval = subject_folds(df_trainval, k=5, seed=SEED)

# Guardamos los datos preprocesados para importarlos
@save "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval
println("Datos procesados y guardados en 'datos_procesados.jld2'")

Datos procesados y guardados en 'datos_procesados.jld2'
